<a href="https://colab.research.google.com/github/sileysa/Semester_5/blob/main/JS09/JS09_Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install library

In [1]:
!pip install -q annoy faiss-cpu hnswlib psutil pandas numpy matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import psutil
import os

from annoy import AnnoyIndex
import faiss
import hnswlib

print("Semua library berhasil diimport!")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS:", faiss.__version__)

Semua library berhasil diimport!
NumPy: 2.4.6
Pandas: 2.2.3
FAISS: 1.15.1


# Membuat dataset

In [3]:
# Menentukan jumlah data dan dimensi
jumlah_data = [1000, 1000000]
dimensi_data = [2, 5]

# Membuat dataset
datasets = {}

np.random.seed(42)

for jumlah in jumlah_data:
    for dimensi in dimensi_data:

        data = np.random.random((jumlah, dimensi)).astype('float32')

        datasets[(jumlah, dimensi)] = data

        print(f"Jumlah data : {jumlah:,}")
        print(f"Dimensi     : {dimensi}D")
        print(f"Shape       : {data.shape}")
        print("-" * 40)

Jumlah data : 1,000
Dimensi     : 2D
Shape       : (1000, 2)
----------------------------------------
Jumlah data : 1,000
Dimensi     : 5D
Shape       : (1000, 5)
----------------------------------------
Jumlah data : 1,000,000
Dimensi     : 2D
Shape       : (1000000, 2)
----------------------------------------
Jumlah data : 1,000,000
Dimensi     : 5D
Shape       : (1000000, 5)
----------------------------------------


# Eksperimen ANNOY

In [4]:
from annoy import AnnoyIndex
import numpy as np
import time
import os
import gc

# Parameter pengujian
K = 10
JUMLAH_QUERY = 100
N_TREES = 10

# Menghitung jarak sebenarnya
def hitung_jarak(data, query, metric):

    if metric == 'euclidean':
        jarak = np.sum((data - query) ** 2, axis=1)

    elif metric == 'manhattan':
        jarak = np.sum(np.abs(data - query), axis=1)

    elif metric == 'cosine':
        dot = np.dot(data, query)
        norm = np.linalg.norm(data, axis=1) * np.linalg.norm(query)
        jarak = 1 - dot / np.maximum(norm, 1e-10)

    return jarak


# Pengujian ANNOY
def uji_annoy(data, metric):

    jumlah, dimensi = data.shape

    # Menyesuaikan metric ANNOY
    metric_annoy = {
        'euclidean': 'euclidean',
        'manhattan': 'manhattan',
        'cosine': 'angular'
    }

    index = AnnoyIndex(dimensi, metric_annoy[metric])

    # Memasukkan data
    for i, vector in enumerate(data):
        index.add_item(i, vector)

    # Mengukur waktu pembuatan index
    start = time.perf_counter()
    index.build(N_TREES)
    build_time = time.perf_counter() - start

    # Membuat query acak
    rng = np.random.default_rng(123)
    queries = rng.random((JUMLAH_QUERY, dimensi)).astype('float32')

    query_time = 0
    recalls = []

    for query in queries:

        # Pencarian ANNOY
        start = time.perf_counter()

        hasil = index.get_nns_by_vector(
            query.tolist(), K
        )

        query_time += time.perf_counter() - start

        # Menghitung hasil pencarian sebenarnya
        jarak = hitung_jarak(data, query, metric)

        ground_truth = np.argpartition(jarak, K)[:K]

        # Menghitung Recall@10
        recall = len(set(hasil) & set(ground_truth)) / K
        recalls.append(recall)

    # Menyimpan ukuran index
    nama_file = f'annoy_{jumlah}_{dimensi}_{metric}.ann'
    index.save(nama_file)

    ukuran_index = os.path.getsize(nama_file) / (1024 ** 2)

    # Membersihkan index
    del index
    gc.collect()

    return {
        'Build Time (s)': build_time,
        'Query Time (ms)': query_time / JUMLAH_QUERY * 1000,
        'Index Size (MB)': ukuran_index,
        'Recall@10': np.mean(recalls)
    }

## Mencoba ANNOY pada dataset kecil

In [5]:
data = datasets[(1000, 2)]

hasil_annoy = []

for metric in ['euclidean', 'manhattan', 'cosine']:

    print(f"Menguji ANNOY - {metric}")

    hasil = uji_annoy(data, metric)

    hasil['Dataset'] = '1000/2D'
    hasil['Metric'] = metric

    hasil_annoy.append(hasil)

    print(hasil)
    print("-" * 40)

Menguji ANNOY - euclidean
{'Build Time (s)': 0.032867632999966645, 'Query Time (ms)': 0.06028495000180101, 'Index Size (MB)': 0.17346954345703125, 'Recall@10': np.float64(0.9940000000000001), 'Dataset': '1000/2D', 'Metric': 'euclidean'}
----------------------------------------
Menguji ANNOY - manhattan
{'Build Time (s)': 0.039192404999994324, 'Query Time (ms)': 0.04132918000209429, 'Index Size (MB)': 0.17456817626953125, 'Recall@10': np.float64(0.995), 'Dataset': '1000/2D', 'Metric': 'manhattan'}
----------------------------------------
Menguji ANNOY - cosine
{'Build Time (s)': 0.055812594999963494, 'Query Time (ms)': 0.0399282399985168, 'Index Size (MB)': 0.14627456665039062, 'Recall@10': np.float64(0.9990000000000001), 'Dataset': '1000/2D', 'Metric': 'cosine'}
----------------------------------------


## Menampilkan hasil

In [6]:
df_annoy = pd.DataFrame(hasil_annoy)

df_annoy = df_annoy[
    ['Dataset', 'Metric', 'Build Time (s)',
     'Query Time (ms)', 'Index Size (MB)', 'Recall@10']
]

display(df_annoy.round(4))

,Dataset,Metric,Build Time (s),Query Time (ms),Index Size (MB),Recall@10
0,1000/2D,euclidean,0.0329,0.0603,0.1735,0.994
1,1000/2D,manhattan,0.0392,0.0413,0.1746,0.995
2,1000/2D,cosine,0.0558,0.0399,0.1463,0.999


## Melanjutkan pengujian ANNOY

In [7]:
# Dataset yang belum diuji
dataset_lanjutan = [
    (1000, 5),
    (1000000, 2),
    (1000000, 5)
]

for jumlah, dimensi in dataset_lanjutan:

    data = datasets[(jumlah, dimensi)]

    for metric in ['euclidean', 'manhattan', 'cosine']:

        print(f"Menguji ANNOY: {jumlah:,} data / {dimensi}D / {metric}")

        hasil = uji_annoy(data, metric)

        hasil['Dataset'] = f'{jumlah}/{dimensi}D'
        hasil['Metric'] = metric

        hasil_annoy.append(hasil)

        print(hasil)
        print("-" * 50)

Menguji ANNOY: 1,000 data / 5D / euclidean
{'Build Time (s)': 0.016190274999985377, 'Query Time (ms)': 0.029357199997548378, 'Index Size (MB)': 0.1696014404296875, 'Recall@10': np.float64(0.9590000000000002), 'Dataset': '1000/5D', 'Metric': 'euclidean'}
--------------------------------------------------
Menguji ANNOY: 1,000 data / 5D / manhattan
{'Build Time (s)': 0.015420912999957181, 'Query Time (ms)': 0.027652210003452637, 'Index Size (MB)': 0.169189453125, 'Recall@10': np.float64(0.936), 'Dataset': '1000/5D', 'Metric': 'manhattan'}
--------------------------------------------------
Menguji ANNOY: 1,000 data / 5D / cosine
{'Build Time (s)': 0.023989027000084207, 'Query Time (ms)': 0.02921367999647373, 'Index Size (MB)': 0.153411865234375, 'Recall@10': np.float64(0.9710000000000001), 'Dataset': '1000/5D', 'Metric': 'cosine'}
--------------------------------------------------
Menguji ANNOY: 1,000,000 data / 2D / euclidean
{'Build Time (s)': 27.987225176000038, 'Query Time (ms)': 0.118

## Menampilkan seluruh hasil ANNOY

In [8]:
df_annoy = pd.DataFrame(hasil_annoy)

df_annoy = df_annoy[
    ['Dataset', 'Metric', 'Build Time (s)',
     'Query Time (ms)', 'Index Size (MB)', 'Recall@10']
]

display(df_annoy.round(4))

,Dataset,Metric,Build Time (s),Query Time (ms),Index Size (MB),Recall@10
0,1000/2D,euclidean,0.0329,0.0603,0.1735,0.994
1,1000/2D,manhattan,0.0392,0.0413,0.1746,0.995
2,1000/2D,cosine,0.0558,0.0399,0.1463,0.999
3,1000/5D,euclidean,0.0162,0.0294,0.1696,0.959
4,1000/5D,manhattan,0.0154,0.0277,0.1692,0.936
5,1000/5D,cosine,0.0240,0.0292,0.1534,0.971
6,1000000/2D,euclidean,27.9872,0.1181,174.0334,0.998
7,1000000/2D,manhattan,29.7048,0.1526,173.6129,0.991
8,1000000/2D,cosine,104.4538,0.1662,147.2736,0.116
9,1000000/5D,euclidean,23.2958,0.1356,169.4256,0.944


# Membuat fungsi pengujian FAISS

In [9]:
# Fungsi pengujian FAISS

def uji_faiss(data, metric):

    jumlah, dimensi = data.shape

    # Menentukan metric FAISS
    metric_faiss = {
        'euclidean': faiss.METRIC_L2,
        'manhattan': faiss.METRIC_L1,
        'cosine': faiss.METRIC_INNER_PRODUCT
    }

    # Persiapan data
    data_faiss = data.copy()

    if metric == 'cosine':
        faiss.normalize_L2(data_faiss)

    # Menentukan jumlah cluster
    nlist = 10 if jumlah == 1000 else 1000
    nprobe = int(np.sqrt(nlist))

    # Membuat quantizer
    quantizer = faiss.IndexFlat(dimensi, metric_faiss[metric])

    # Membuat index IVF
    index = faiss.IndexIVFFlat(
        quantizer,
        dimensi,
        nlist,
        metric_faiss[metric]
    )

    # Mengukur waktu pembangunan index
    start = time.perf_counter()

    # Training menggunakan sampel data
    if jumlah > 100000:
        rng = np.random.default_rng(42)
        sample_idx = rng.choice( jumlah, 100000, replace=False)
        train_data = data_faiss[sample_idx]
    else:
        train_data = data_faiss

    index.train(train_data)
    index.add(data_faiss)

    build_time = time.perf_counter() - start

    # Mengatur jumlah cluster yang diperiksa
    index.nprobe = nprobe

    # Membuat query
    rng = np.random.default_rng(123)
    queries = rng.random(
        (JUMLAH_QUERY, dimensi)
    ).astype('float32')

    if metric == 'cosine':
        faiss.normalize_L2(queries)

    query_time = 0
    recalls = []

    # Melakukan pencarian
    for query in queries:

        start = time.perf_counter()

        _, hasil = index.search(
            query.reshape(1, -1), K
        )

        query_time += time.perf_counter() - start

        # Menghitung ground truth
        jarak = hitung_jarak(data_faiss, query, metric)

        ground_truth = np.argpartition(jarak, K)[:K]

        recall = len(
            set(hasil[0]) & set(ground_truth)
        ) / K

        recalls.append(recall)

    # Menghitung ukuran index
    ukuran_index = (
        faiss.serialize_index(index).nbytes
        / (1024 ** 2)
    )

    del index
    gc.collect()

    return {
        'Build Time (s)': build_time,
        'Query Time (ms)': query_time / JUMLAH_QUERY * 1000,
        'Index Size (MB)': ukuran_index,
        'Recall@10': np.mean(recalls)
    }

# Menguji FAISS pada dataset 1.000/2D

In [10]:
hasil_faiss = []

data = datasets[(1000, 2)]

for metric in ['euclidean', 'manhattan', 'cosine']:

    print(f"Menguji FAISS - {metric}")

    hasil = uji_faiss(data, metric)

    hasil['Dataset'] = '1000/2D'
    hasil['Metric'] = metric

    hasil_faiss.append(hasil)

    print(hasil)
    print("-" * 40)

Menguji FAISS - euclidean
{'Build Time (s)': 0.05311258300002919, 'Query Time (ms)': 0.05517948999795408, 'Index Size (MB)': 0.015543937683105469, 'Recall@10': np.float64(0.9990000000000001), 'Dataset': '1000/2D', 'Metric': 'euclidean'}
----------------------------------------
Menguji FAISS - manhattan
{'Build Time (s)': 0.002099192000059702, 'Query Time (ms)': 0.28378960999134506, 'Index Size (MB)': 0.015551567077636719, 'Recall@10': np.float64(1.0), 'Dataset': '1000/2D', 'Metric': 'manhattan'}
----------------------------------------
Menguji FAISS - cosine
{'Build Time (s)': 0.0011207609998109547, 'Query Time (ms)': 0.028468780008097383, 'Index Size (MB)': 0.015543937683105469, 'Recall@10': np.float64(0.998), 'Dataset': '1000/2D', 'Metric': 'cosine'}
----------------------------------------


# Menampilkan hasil

In [11]:
df_faiss = pd.DataFrame(hasil_faiss)

df_faiss = df_faiss[
    ['Dataset', 'Metric', 'Build Time (s)',
     'Query Time (ms)', 'Index Size (MB)', 'Recall@10']
]

display(df_faiss.round(4))

,Dataset,Metric,Build Time (s),Query Time (ms),Index Size (MB),Recall@10
0,1000/2D,euclidean,0.0531,0.0552,0.0155,0.999
1,1000/2D,manhattan,0.0021,0.2838,0.0156,1.000
2,1000/2D,cosine,0.0011,0.0285,0.0155,0.998


# Melanjutkan pengujian FAISS

In [12]:
# Dataset yang belum diuji
dataset_lanjutan = [
    (1000, 5),
    (1000000, 2),
    (1000000, 5)
]

for jumlah, dimensi in dataset_lanjutan:

    data = datasets[(jumlah, dimensi)]

    for metric in ['euclidean', 'manhattan', 'cosine']:

        print(f"Menguji FAISS: {jumlah:,} data / {dimensi}D / {metric}")

        hasil = uji_faiss(data, metric)

        hasil['Dataset'] = f'{jumlah}/{dimensi}D'
        hasil['Metric'] = metric

        hasil_faiss.append(hasil)

        print(hasil)
        print("-" * 50)

Menguji FAISS: 1,000 data / 5D / euclidean
{'Build Time (s)': 0.005687178000016502, 'Query Time (ms)': 0.03226762000167582, 'Index Size (MB)': 0.02710247039794922, 'Recall@10': np.float64(0.9720000000000002), 'Dataset': '1000/5D', 'Metric': 'euclidean'}
--------------------------------------------------
Menguji FAISS: 1,000 data / 5D / manhattan
{'Build Time (s)': 0.0015584069999476924, 'Query Time (ms)': 0.27834635999852253, 'Index Size (MB)': 0.02711009979248047, 'Recall@10': np.float64(0.9450000000000002), 'Dataset': '1000/5D', 'Metric': 'manhattan'}
--------------------------------------------------
Menguji FAISS: 1,000 data / 5D / cosine
{'Build Time (s)': 0.004238453000198206, 'Query Time (ms)': 0.029650650001258327, 'Index Size (MB)': 0.02710247039794922, 'Recall@10': np.float64(0.9740000000000001), 'Dataset': '1000/5D', 'Metric': 'cosine'}
--------------------------------------------------
Menguji FAISS: 1,000,000 data / 2D / euclidean
{'Build Time (s)': 0.5528573400001733, 'Qu

# Menampilkan seluruh hasil FAISS

In [13]:
df_faiss = pd.DataFrame(hasil_faiss)

df_faiss = df_faiss[
    ['Dataset', 'Metric', 'Build Time (s)',
     'Query Time (ms)', 'Index Size (MB)', 'Recall@10']
]

display(df_faiss.round(4))

,Dataset,Metric,Build Time (s),Query Time (ms),Index Size (MB),Recall@10
0,1000/2D,euclidean,0.0531,0.0552,0.0155,0.999
1,1000/2D,manhattan,0.0021,0.2838,0.0156,1.000
2,1000/2D,cosine,0.0011,0.0285,0.0155,0.998
3,1000/5D,euclidean,0.0057,0.0323,0.0271,0.972
4,1000/5D,manhattan,0.0016,0.2783,0.0271,0.945
5,1000/5D,cosine,0.0042,0.0297,0.0271,0.974
6,1000000/2D,euclidean,0.5529,0.5637,15.2742,1.000
7,1000000/2D,manhattan,16.1693,0.7901,15.2742,1.000
8,1000000/2D,cosine,2.4515,0.9632,15.2742,0.090
9,1000000/5D,euclidean,0.7580,0.5005,26.7297,1.000


# Membuat fungsi pengujian HNSW

In [21]:
def uji_hnsw(data, metric):
    jumlah, dimensi = data.shape

    # Menentukan metric HNSW
    metric_hnsw = {
        'euclidean': 'l2',
        'cosine': 'cosine'
    }

    if metric not in metric_hnsw:
        raise ValueError(f"HNSW tidak mendukung metric {metric}")

    # Membuat index HNSW
    index = hnswlib.Index(
        space=metric_hnsw[metric],
        dim=dimensi
    )

    index.init_index(
        max_elements=jumlah,
        ef_construction=200,
        M=16,
        random_seed=42
    )

    index.set_num_threads(4)

    # Mengukur waktu pembangunan index
    start = time.perf_counter()

    index.add_items(data, np.arange(jumlah))

    build_time = time.perf_counter() - start

    # Mengatur kualitas pencarian
    index.set_ef(200)

    # Membuat data query
    rng = np.random.default_rng(123)

    queries = rng.random(
        (JUMLAH_QUERY, dimensi)
    ).astype('float32')

    query_time = 0
    recalls = []

    # Menguji pencarian
    for query in queries:

        start = time.perf_counter()

        hasil, _ = index.knn_query(
            query.reshape(1, -1),
            k=K,
            num_threads=1
        )

        query_time += time.perf_counter() - start

        # Menghitung ground truth
        jarak = hitung_jarak(data, query, metric)

        ground_truth = np.argpartition(jarak, K)[:K]

        # Menghitung Recall@10
        recall = len(
            set(hasil[0]) & set(ground_truth)
        ) / K

        recalls.append(recall)

    # Menyimpan index untuk menghitung ukuran
    nama_file = f'hnsw_{jumlah}_{dimensi}_{metric}.bin'

    index.save_index(nama_file)

    ukuran_index = os.path.getsize(nama_file) / (1024 ** 2)

    # Menghapus index dari memori
    del index
    gc.collect()

    return {
        'Build Time (s)': build_time,
        'Query Time (ms)': query_time / JUMLAH_QUERY * 1000,
        'Index Size (MB)': ukuran_index,
        'Recall@10': np.mean(recalls)
    }

# Menjalankan pengujian seluruh dataset

In [22]:
hasil_hnsw = []

for (jumlah, dimensi), data in datasets.items():

    for metric in ['euclidean', 'cosine']:

        print(
            f"Menguji HNSW: {jumlah:,} data / "
            f"{dimensi}D / {metric}"
        )

        hasil = uji_hnsw(data, metric)

        hasil['Dataset'] = f'{jumlah}/{dimensi}D'
        hasil['Metric'] = metric

        hasil_hnsw.append(hasil)

        print(hasil)
        print('-' * 50)

Menguji HNSW: 1,000 data / 2D / euclidean
{'Build Time (s)': 0.5216243529998792, 'Query Time (ms)': 0.3287919900003544, 'Index Size (MB)': 0.14939498901367188, 'Recall@10': np.float64(1.0), 'Dataset': '1000/2D', 'Metric': 'euclidean'}
--------------------------------------------------
Menguji HNSW: 1,000 data / 2D / cosine
{'Build Time (s)': 0.10647437999978138, 'Query Time (ms)': 0.1227985600144166, 'Index Size (MB)': 0.14939498901367188, 'Recall@10': np.float64(0.9990000000000001), 'Dataset': '1000/2D', 'Metric': 'cosine'}
--------------------------------------------------
Menguji HNSW: 1,000 data / 5D / euclidean
{'Build Time (s)': 0.09552882500020132, 'Query Time (ms)': 0.08890053001778142, 'Index Size (MB)': 0.16083908081054688, 'Recall@10': np.float64(1.0), 'Dataset': '1000/5D', 'Metric': 'euclidean'}
--------------------------------------------------
Menguji HNSW: 1,000 data / 5D / cosine
{'Build Time (s)': 0.058734479999657196, 'Query Time (ms)': 0.08880475000751176, 'Index Siz

# Menampilkan tabel hasil pengujian

In [23]:
df_hnsw = pd.DataFrame(hasil_hnsw)

df_hnsw = df_hnsw[
    [
        'Dataset',
        'Metric',
        'Build Time (s)',
        'Query Time (ms)',
        'Index Size (MB)',
        'Recall@10'
    ]
]

display(df_hnsw.round(4))

,Dataset,Metric,Build Time (s),Query Time (ms),Index Size (MB),Recall@10
0,1000/2D,euclidean,0.5216,0.3288,0.1494,1.000
1,1000/2D,cosine,0.1065,0.1228,0.1494,0.999
2,1000/5D,euclidean,0.0955,0.0889,0.1608,1.000
3,1000/5D,cosine,0.0587,0.0888,0.1608,1.000
4,1000000/2D,euclidean,130.2333,0.2162,149.2759,1.000
5,1000000/2D,cosine,164.8318,0.4202,149.2757,0.062
6,1000000/5D,euclidean,198.9555,0.3677,160.7199,1.000
7,1000000/5D,cosine,178.3281,0.5666,160.7199,1.000


In [24]:
# Mengambil dataset 1 juta data / 2D
data = datasets[(1000000, 2)]

# Membuat satu query
rng = np.random.default_rng(123)
query = rng.random(2).astype('float32')

# Menghitung jarak cosine seluruh data
jarak = hitung_jarak(data, query, 'cosine')

# Mengambil 10 tetangga terdekat secara exact
ground_truth = np.argsort(jarak)[:10]

print("Query:", query)
print("\n10 tetangga terdekat (Ground Truth):")
print(ground_truth)
print("\nNilai jarak cosine:")
print(jarak[ground_truth])

Query: [0.6823519  0.05382102]

10 tetangga terdekat (Ground Truth):
[636494 192379 705071 737902 600159 383547 323733 990034 481572 462742]

Nilai jarak cosine:
[-1.1920929e-07 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07
 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07
 -1.1920929e-07 -1.1920929e-07]


In [25]:
# Menghitung jumlah nilai jarak cosine yang unik
jumlah_unik = len(np.unique(jarak))

print("Jumlah seluruh data:", len(jarak))
print("Jumlah nilai jarak unik:", jumlah_unik)

# Melihat distribusi nilai jarak terdekat
print("\n20 nilai jarak terkecil:")
print(np.sort(jarak)[:20])

Jumlah seluruh data: 1000000
Jumlah nilai jarak unik: 919899

20 nilai jarak terkecil:
[-1.1920929e-07 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07
 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07
 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07
 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07
 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07]


In [26]:
# Membuat ulang index HNSW
index = hnswlib.Index(space='cosine', dim=2)

index.load_index('hnsw_1000000_2_cosine.bin')
index.set_ef(200)

# Melakukan pencarian
hasil, jarak_hnsw = index.knn_query(
    query.reshape(1, -1),
    k=10
)

print("Hasil pencarian HNSW:")
print(hasil[0])

print("\nJarak Cosine HNSW:")
print(jarak_hnsw[0])

print("\nGround Truth:")
print(ground_truth)

Hasil pencarian HNSW:
[ 52102 413700 697353 111472 486305 571385 633493 673027 753283 981701]

Jarak Cosine HNSW:
[-1.1920929e-07 -1.1920929e-07 -1.1920929e-07  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00]

Ground Truth:
[636494 192379 705071 737902 600159 383547 323733 990034 481572 462742]


In [27]:
# Menghitung jarak cosine untuk hasil HNSW
jarak_hasil_hnsw = hitung_jarak(
    data[hasil[0]],
    query,
    'cosine'
)

# Menghitung jarak cosine untuk Ground Truth
jarak_ground_truth = hitung_jarak(
    data[ground_truth],
    query,
    'cosine'
)

print("Rata-rata jarak Ground Truth:",
      np.mean(jarak_ground_truth))

print("Rata-rata jarak HNSW:",
      np.mean(jarak_hasil_hnsw))

print("\nJarak Ground Truth:")
print(jarak_ground_truth)

print("\nJarak HNSW:")
print(jarak_hasil_hnsw)

Rata-rata jarak Ground Truth: -1.1920929e-07
Rata-rata jarak HNSW: -2.3841858e-08

Jarak Ground Truth:
[-1.1920929e-07 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07
 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07 -1.1920929e-07
 -1.1920929e-07 -1.1920929e-07]

Jarak HNSW:
[ 0.0000000e+00 -1.1920929e-07 -1.1920929e-07  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00]


# Menggabungkan hasil pengujian

In [29]:
tabel_perbandingan = df_perbandingan.pivot(
    index=['Dataset', 'Metric'],
    columns='Algoritma',
    values=[
        'Build Time (s)',
        'Query Time (ms)',
        'Index Size (MB)',
        'Recall@10'
    ]
)

display(tabel_perbandingan.round(4))

Build Time (s)                    Query Time (ms)  \
Algoritma                     ANNOY    FAISS      HNSW           ANNOY   
Dataset    Metric                                                        
1000/2D    cosine            0.0558   0.0011    0.1065          0.0399   
           euclidean         0.0329   0.0531    0.5216          0.0603   
           manhattan         0.0392   0.0021       NaN          0.0413   
1000/5D    cosine            0.0240   0.0042    0.0587          0.0292   
           euclidean         0.0162   0.0057    0.0955          0.0294   
           manhattan         0.0154   0.0016       NaN          0.0277   
1000000/2D cosine          104.4538   2.4515  164.8318          0.1662   
           euclidean        27.9872   0.5529  130.2333          0.1181   
           manhattan        29.7048  16.1693       NaN          0.1526   
1000000/5D cosine           29.1511   3.5175  178.3281          0.1280   
           euclidean        23.2958   0.7580  198.9555          0.1356   
           manhattan        21.6703  15.6350       NaN          0.1242   

                                     Index Size (MB)                     \
Algoritma              FAISS    HNSW           ANNOY    FAISS      HNSW   
Dataset    Metric                                                         
1000/2D    cosine     0.0285  0.1228          0.1463   0.0155    0.1494   
           euclidean  0.0552  0.3288          0.1735   0.0155    0.1494   
           manhattan  0.2838     NaN          0.1746   0.0156       NaN   
1000/5D    cosine     0.0297  0.0888          0.1534   0.0271    0.1608   
           euclidean  0.0323  0.0889          0.1696   0.0271    0.1608   
           manhattan  0.2783     NaN          0.1692   0.0271       NaN   
1000000/2D cosine     0.9632  0.4202        147.2736  15.2742  149.2757   
           euclidean  0.5637  0.2162        174.0334  15.2742  149.2759   
           manhattan  0.7901     NaN        173.6129  15.2742       NaN   
1000000/5D cosine     0.7826  0.5666        152.4105  26.7297  160.7199   
           euclidean  0.5005  0.3677        169.4256  26.7297  160.7199   
           manhattan  0.9165     NaN        169.2219  26.7297       NaN   

                     Recall@10                
Algoritma                ANNOY  FAISS   HNSW  
Dataset    Metric                             
1000/2D    cosine        0.999  0.998  0.999  
           euclidean     0.994  0.999  1.000  
           manhattan     0.995  1.000    NaN  
1000/5D    cosine        0.971  0.974  1.000  
           euclidean     0.959  0.972  1.000  
           manhattan     0.936  0.945    NaN  
1000000/2D cosine        0.116  0.090  0.062  
           euclidean     0.998  1.000  1.000  
           manhattan     0.991  1.000    NaN  
1000000/5D cosine        0.966  1.000  1.000  
           euclidean     0.944  1.000  1.000  
           manhattan     0.948  1.000    NaN